In [ ]:
import requests
from bs4 import BeautifulSoup
print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
import requests
url="https://books.toscrape.com/"
response = requests.get(url)
print(response.status_code)

200


In [ ]:
response = requests.get(url)

In [ ]:
response.text

'<!DOCTYPE html>\n<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->\n<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->\n<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->\n<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->\n    <head>\n        <title>\n    All products | Books to Scrape - Sandbox\n</title>\n\n        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />\n        <meta name="created" content="24th Jun 2016 09:29" />\n        <meta name="description" content="" />\n        <meta name="viewport" content="width=device-width" />\n        <meta name="robots" content="NOARCHIVE,NOCACHE" />\n\n        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->\n        <!--[if lt IE 9]>\n        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>\n        <![endif]-->\n\n        \n            <link rel="shortcut icon" href

In [ ]:
print(response.text[:500])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
print(type(soup))

<class 'bs4.BeautifulSoup'>


In [ ]:
print(soup.title.text)
products = soup.find_all("article",class_="product_pod")
print(len(products))
first_product = products[0]
print(first_product.h3.a["title"])

price = first_product.find("p",class_="price_color").text
print(price)


    All products | Books to Scrape - Sandbox

20
A Light in the Attic
Â£51.77


In [ ]:
for product in products:
  name = product.h3.a["title"]
  price = product.find("p", class_="price_color").text
  print("Book:",name)
  print("Price:",price)
  print("-"*40)

Book: A Light in the Attic
Price: Â£51.77
----------------------------------------
Book: Tipping the Velvet
Price: Â£53.74
----------------------------------------
Book: Soumission
Price: Â£50.10
----------------------------------------
Book: Sharp Objects
Price: Â£47.82
----------------------------------------
Book: Sapiens: A Brief History of Humankind
Price: Â£54.23
----------------------------------------
Book: The Requiem Red
Price: Â£22.65
----------------------------------------
Book: The Dirty Little Secrets of Getting Your Dream Job
Price: Â£33.34
----------------------------------------
Book: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Price: Â£17.93
----------------------------------------
Book: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
Price: Â£22.60
----------------------------------------
Book: The Black Maria
Price: Â£52.15
----------------------------------------
Book: Starv

In [ ]:
!pip install google-colab-selenium

In [ ]:
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

# Initialize the driver (this handles headless, no-sandbox, and dev-shm-usage automatically)
driver = gs.Chrome()

url = "https://books.toscrape.com/"
books_data = []

try:
    driver.get(url)

    while True:
        # Wait for the book cards to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "product_pod"))
        )

        # Find all books on the current page
        books = driver.find_elements(By.CLASS_NAME, "product_pod")

        for book in books:
            title = book.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            price = book.find_element(By.CLASS_NAME, "price_color").text
            books_data.append({"Title": title, "Price": price})

        # Check for 'Next' button to navigate
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "li.next > a")
            next_button.click()
        except:
            print("Scraping complete. No more pages.")
            break

    # Convert to a Pandas DataFrame for easy viewing in Colab
    df = pd.DataFrame(books_data)
    display(df.head(10))

finally:
    driver.quit()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Scraping complete. No more pages.


,Title,Price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23
5,The Requiem Red,£22.65
6,The Dirty Little Secrets of Getting Your Dream...,£33.34
7,The Coming Woman: A Novel Based on the Life of...,£17.93
8,The Boys in the Boat: Nine Americans and Their...,£22.60
9,The Black Maria,£52.15


In [ ]:
import requests
from lxml import html
import pandas as pd

base_url = "https://books.toscrape.com/"
current_page_url = "https://books.toscrape.com/index.html"
books_data = []

while current_page_url:
    # 1. Fetch the page content
    response = requests.get(current_page_url)

    # 2. Convert the HTML content into an lxml tree
    tree = html.fromstring(response.content)

    # 3. Find all book containers using XPath
    # In lxml, //article[@class='product_pod'] finds the book blocks
    books = tree.xpath("//article[@class='product_pod']")

    for book in books:
        # Extract title: h3 > a (title attribute)
        title = book.xpath(".//h3/a/@title")[0]

        # Extract price: p[@class='price_color']
        price = book.xpath(".//p[@class='price_color']/text()")[0]

        books_data.append({"Title": title, "Price": price})

    # 4. Handle Pagination
    # Look for the 'next' button link
    next_rel_url = tree.xpath("//li[@class='next']/a/@href")

    if next_rel_url:
        # Since the 'next' URL is relative (e.g., 'catalogue/page-2.html'),
        # we need to handle the URL formatting
        if "catalogue/" in next_rel_url[0]:
            current_page_url = base_url + next_rel_url[0]
        else:
            current_page_url = base_url + "catalogue/" + next_rel_url[0]
    else:
        current_page_url = None

# Results
df = pd.DataFrame(books_data)
print(f"Total books scraped: {len(df)}")
display(df.head(10))

Total books scraped: 1000


,Title,Price
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23
5,The Requiem Red,£22.65
6,The Dirty Little Secrets of Getting Your Dream...,£33.34
7,The Coming Woman: A Novel Based on the Life of...,£17.93
8,The Boys in the Boat: Nine Americans and Their...,£22.60
9,The Black Maria,£52.15
